In [1]:
import pandas as pd

import sys
sys.path.insert(1, '../../scripts')
from utils.load_environmental_variables import build_files_path, input_data_path
prebuild = '/data2/hratch/human_me/prebuild/'

Format

In [104]:
ptr = pd.read_csv(prebuild + 'PTR_Gagneur_unprocessed.tsv', sep = '\t')
ptr.replace('   NA', float('nan'), inplace = True)
ptr.replace('    NA', float('nan'), inplace = True)

ptr_cols = [col for col in ptr.columns if 'PTR' in col]
for col in ptr_cols:
    ptr[col] = ptr[col].astype(float)

ptr = pd.concat([ptr[['EnsemblGeneID']], 10**ptr[ptr_cols]], axis = 1)

ptr.set_index('EnsemblGeneID', inplace = True)
print('Median PTR value across all tissues, before ID mapping: {:.4f}'.format(ptr.median(axis = 1).median()))

Median PTR value across all tissues, before ID mapping: 65162.8394


In [26]:
# mapping
# preserve original psim ids

# conserve existing mapping (consider many to one)
psim_me = pd.read_hdf('/data2/hratch/human_me/temp_psim.h5', key = 'n_exons') # ok to use final psim also
temp = psim_me[psim_me.ENSG_ID.isin(ptr.index)][['ENSG_ID', 'HGNC_ID']]
cond = temp.ENSG_ID.unique().shape[0] == temp.HGNC_ID.unique().shape[0] == temp.shape[0]
if cond:
    mapper = dict(zip(temp.ENSG_ID, temp.HGNC_ID))
else: 
    # use protein_turnover script to adapt in this case
    raise ValueError('Was working with 1-to-1 mapping before')
del temp



In [82]:
ptr_ids = pd.DataFrame(data = {'ENSG_ID': ptr.index, 'HGNC_ID': ptr.index.map(mapper)})
p1 = ptr_ids[ids.HGNC_ID.notna()]
p2 = ptr_ids[ids.HGNC_ID.isna()]

ehm = pd.read_csv(prebuild + 'sequence_information/identifiers.txt', sep = '\t')
ehm = ehm[ehm['Ensembl gene ID'].notna() & ehm['HGNC ID'].notna() & ehm['Ensembl gene ID'].isin(p2.ENSG_ID)]
ehm = ehm[['HGNC ID', 'Ensembl gene ID']]
ehm.drop_duplicates(inplace = True)

cond = ehm['HGNC ID'].unique().shape[0] == ehm['Ensembl gene ID'].unique().shape[0] == ehm.shape[0]
if not cond:
    raise ValueError('Expected 1 to 1 mapping, consider expanding for redundant HGNCs')
p2['HGNC_ID'] = p2.ENSG_ID.map(dict(zip(ehm['Ensembl gene ID'], ehm['HGNC ID'])))

ptr_ids = pd.concat([p1, p2], axis = 0, ignore_index=True)

ptr['HGNC_ID'] = ptr.index.map(dict(zip(ptr_ids.ENSG_ID, ptr_ids.HGNC_ID)))
ptr['ENSG_ID'] = ptr.index
ptr.reset_index(inplace = True, drop = True)
ptr = pd.concat([ptr.iloc[:,-2:], ptr.iloc[:,:-2]], axis = 1)
ptr.to_csv(build_files_path + 'PTR_Gagneur_processed.tsv', sep = '\t')